<a href="https://colab.research.google.com/github/anantshri1/homo-lumo-gat/blob/main/notebooks/01_data_and_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Imports, Initialization**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# All our saved files go here — change if you want a different folder
PROJECT_DIR = '/content/drive/MyDrive/homo-lumo-gat'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project dir: {PROJECT_DIR}")


Mounted at /content/drive
Project dir: /content/drive/MyDrive/homo-lumo-gat


In [2]:
import torch

# Print torch version first — PyG install depends on this
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [3]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00


`rdkit` is the open-source toolkit for cheminformatics.

In [4]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 24.3 MB/s eta 0:00:00


A few things worth noting:

* `GATv2Conv` is the actual graph attention layer from `PyG`
* `global_mean_pool` and `global_max_pool` are the pooling operations from our architecture diagram — we'll concatenate both
* `QM9` is `PyG`'s built-in dataset loader — it handles downloading and caching automatically

In [5]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import QM9
import rdkit
from rdkit import Chem
import numpy as np
from tqdm import tqdm

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"RDKit: {rdkit.__version__}")

Torch: 2.11.0+cu128
CUDA available: True
Device: Tesla T4
RDKit: 2026.03.3


# **Data Loading**

In [6]:
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Training config — all in one place so easy to tweak
CONFIG = {
    'target_idx': 4,       # QM9 target 4 = HOMO-LUMO gap
    'batch_size': 32,
    'lr': 1e-3,
    'epochs': 100,
    'hidden_dim': 256,
    'num_heads': 4,
    'dropout': 0.1,
    'seed': 42,
}

torch.manual_seed(CONFIG['seed'])
print("Config:", CONFIG)

Config: {'target_idx': 4, 'batch_size': 32, 'lr': 0.001, 'epochs': 100, 'hidden_dim': 256, 'num_heads': 4, 'dropout': 0.1, 'seed': 42}


`PyG`'s `QM9` class downloads ~130k small organic molecules. Each molecule is already stored as a graph — atoms are nodes, bonds are edges. `PyG` has done the `RDKit` parsing for us here.

In [7]:
from torch_geometric.datasets import QM9
from torch_geometric.transforms import NormalizeFeatures

# PyG downloads and caches QM9 automatically to DATA_DIR
dataset = QM9(root=DATA_DIR)

print(f"Dataset size: {len(dataset)} molecules")
print(f"Number of node features: {dataset.num_node_features}")
print(f"Number of edge features: {dataset.num_edge_features}")
print(f"Number of targets: {dataset.num_classes}")

# Inspect one molecule
mol = dataset[0]
print(f"\nExample molecule:")
print(f"  Nodes (atoms): {mol.num_nodes}")
print(f"  Edges (bonds): {mol.num_edges}")
print(f"  Node feature shape: {mol.x.shape}")
print(f"  Edge feature shape: {mol.edge_attr.shape}")
print(f"  Target shape: {mol.y.shape}")
print(f"  HOMO-LUMO gap (eV): {mol.y[0, CONFIG['target_idx']].item():.4f}")

Dataset size: 130831 molecules
Number of node features: 11
Number of edge features: 4
Number of targets: 19

Example molecule:
  Nodes (atoms): 5
  Edges (bonds): 8
  Node feature shape: torch.Size([5, 11])
  Edge feature shape: torch.Size([8, 4])
  Target shape: torch.Size([1, 19])
  HOMO-LUMO gap (eV): 13.7363


In [8]:
# QM9 has ~130k molecules — standard split in literature
n = len(dataset)
n_train = 110000
n_val = 10000
n_test = n - n_train - n_val

# Shuffle with fixed seed for reproducibility
torch.manual_seed(CONFIG['seed'])
perm = torch.randperm(n)

train_dataset = dataset[perm[:n_train]]
val_dataset = dataset[perm[n_train:n_train + n_val]]
test_dataset = dataset[perm[n_train + n_val:]]

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

# DataLoader batches graphs together efficiently
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"\nBatches per epoch: {len(train_loader)}")

Train: 110000
Val:   10000
Test:  10831

Batches per epoch: 3438


# **Graph Neural Network**

* `GATv2Conv` vs regular `GATConv` — regular `GAT` computes attention weights before combining node features, which means the attention is query-independent. `GATv2` fixes this by computing attention after concatenating and transforming, making it strictly more expressive. This is a 2022 paper improvement.
* `concat=False` — with multiple attention heads, you can either concatenate all head outputs (making the output `hidden_dim × num_heads`) or average them (`concat=False`). We average to keep dimensions manageable across layers.
* Residual connections — same idea as `ResNet`. Without them, gradients vanish in deep GNNs. The `nn.Linear` projections exist because layer 1's input (`in_channels=11`) doesn't match its output (`hidden_dim=256`), so we can't add them directly without projecting first.
* `batch` argument in forward — this is that special tensor `PyG`'s `DataLoader` creates. `global_mean_pool(h3, batch)` averages all atom embeddings that belong to the same molecule, giving one vector per molecule.

In [9]:
class GATv2Model(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_dim, num_heads, dropout):
        super().__init__()

        # 3 GATv2 convolution layers
        self.conv1 = GATv2Conv(in_channels, hidden_dim, heads=num_heads,
                                edge_dim=edge_dim, dropout=dropout, concat=False)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads,
                                edge_dim=edge_dim, dropout=dropout, concat=False)
        self.conv3 = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads,
                                edge_dim=edge_dim, dropout=dropout, concat=False)

        # Residual projections (to match dimensions)
        self.res1 = nn.Linear(in_channels, hidden_dim)
        self.res2 = nn.Linear(hidden_dim, hidden_dim)
        self.res3 = nn.Linear(hidden_dim, hidden_dim)

        # MLP head — input is hidden_dim*2 because we concat mean+max pooling
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        self.dropout = nn.Dropout(dropout)
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr, batch):
        # Layer 1
        h = self.act(self.conv1(x, edge_index, edge_attr))
        h = h + self.res1(x)          # residual connection
        h = self.dropout(h)

        # Layer 2
        h2 = self.act(self.conv2(h, edge_index, edge_attr))
        h2 = h2 + self.res2(h)        # residual connection
        h2 = self.dropout(h2)

        # Layer 3
        h3 = self.act(self.conv3(h2, edge_index, edge_attr))
        h3 = h3 + self.res3(h2)       # residual connection

        # Global pooling — collapses all atom embeddings into one molecule embedding
        mean_pool = global_mean_pool(h3, batch)
        max_pool = global_max_pool(h3, batch)
        h_graph = torch.cat([mean_pool, max_pool], dim=1)

        return self.mlp(h_graph).squeeze(-1)

In [10]:
# QM9 node/edge feature dimensions
in_channels = dataset.num_node_features    # 11
edge_dim = dataset.num_edge_features       # 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = GATv2Model(
    in_channels=in_channels,
    edge_dim=edge_dim,
    hidden_dim=CONFIG['hidden_dim'],
    num_heads=CONFIG['num_heads'],
    dropout=CONFIG['dropout']
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Sanity check — run one batch through
batch = next(iter(train_loader))
batch = batch.to(device)
with torch.no_grad():
    out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
print(f"\nInput batch size: {batch.num_graphs} molecules")
print(f"Output shape: {out.shape}")    # should be [32]
print(f"Sample prediction: {out[0].item():.4f} eV")

Using device: cuda
Trainable parameters: 1,392,385

Input batch size: 32 molecules
Output shape: torch.Size([32])
Sample prediction: 0.0465 eV


* `model.train()` vs `model.eval()` — these switch the model between two modes. The difference matters for dropout: during training, dropout randomly zeros neurons (regularization). During evaluation you want deterministic output, so dropout is disabled. Keras does this automatically; PyTorch makes you call it explicitly.
* `optimizer.zero_grad()` — PyTorch accumulates gradients by default. If you don't zero them before each batch, gradients from the previous batch add to the current one. **Always zero before forward pass.**
The manual training step sequence — this is the core PyTorch loop, always in this order:
```
zero_grad → forward pass → compute loss → backward → step
```

* `@torch.no_grad()` as a decorator — same as with `torch.no_grad()` but cleaner for functions. The evaluate function never needs gradients so we decorate the whole thing.
* L1 loss (MAE) — we use Mean Absolute Error rather than MSE because it's the standard benchmark metric for QM9, and it's less sensitive to outlier molecules.
* `loss.item() * batch.num_graphs` — we accumulate the total loss across all molecules, then divide by dataset size at the end. This gives true average MAE per molecule rather than average of batch averages (which would be slightly wrong for the last batch if it's smaller).

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        target = batch.y[:, CONFIG['target_idx']]

        loss = nn.functional.l1_loss(pred, target)  # MAE loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        target = batch.y[:, CONFIG['target_idx']]
        total_loss += nn.functional.l1_loss(pred, target).item() * batch.num_graphs
    return total_loss / len(loader.dataset)

* What we save in the checkpoint — not just model weights (`model_state_dict`) but also `optimizer_state_dict`. This matters because Adam optimizer has internal momentum state. If you only save weights and your runtime dies, you can resume training but the optimizer starts cold, which hurts convergence for the first few epochs. Saving both means a true resume.
* `best_model.pt` vs `checkpoint_epoch_N.pt` — two different save strategies. Best model is what you'll deploy. Periodic checkpoints are insurance.
* `scheduler.step(val_mae)` — the learning rate scheduler watches validation MAE. If it doesn't improve for 10 epochs (`patience=10`), it halves the learning rate (`factor=0.5`). This is a standard trick to squeeze out the last few points of MAE.

In [12]:
best_val_mae = float('inf')
train_losses = []
val_losses = []

for epoch in range(1, CONFIG['epochs'] + 1):
    train_mae = train_epoch(model, train_loader, optimizer, device)
    val_mae = evaluate(model, val_loader, device)
    scheduler.step(val_mae)

    train_losses.append(train_mae)
    val_losses.append(val_mae)

    # Save best model to Drive
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_mae': val_mae,
            'config': CONFIG,
        }, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))

    # Also save latest checkpoint every 10 epochs (insurance against runtime death)
    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_mae': val_mae,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'config': CONFIG,
        }, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pt'))
        print(f"Epoch {epoch:03d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f} | Best: {best_val_mae:.4f}")

/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 010 | Train MAE: 0.3272 | Val MAE: 0.2684 | Best: 0.2640
Epoch 020 | Train MAE: 0.2694 | Val MAE: 0.2425 | Best: 0.2201
Epoch 030 | Train MAE: 0.2335 | Val MAE: 0.2302 | Best: 0.2134
Epoch 040 | Train MAE: 0.2230 | Val MAE: 0.2167 | Best: 0.2030
Epoch 050 | Train MAE: 0.2164 | Val MAE: 0.2032 | Best: 0.1982
Epoch 060 | Train MAE: 0.2109 | Val MAE: 0.2016 | Best: 0.1913
Epoch 070 | Train MAE: 0.2076 | Val MAE: 0.2153 | Best: 0.1828
Epoch 080 | Train MAE: 0.1924 | Val MAE: 0.1912 | Best: 0.1828
Epoch 090 | Train MAE: 0.1849 | Val MAE: 0.1857 | Best: 0.1799
Epoch 100 | Train MAE: 0.1822 | Val MAE: 0.1779 | Best: 0.1767


In [14]:
# Load best model and evaluate on test set
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_mae = evaluate(model, test_loader, device)
print(f"Test MAE: {test_mae:.4f} eV")
print(f"Best val MAE: {checkpoint['val_mae']:.4f} eV")
print(f"Trained for: {checkpoint['epoch']} epochs")

# Published baselines on QM9 HOMO-LUMO gap:
print(f"\nBaseline comparison:")
print(f"  Ours:    {test_mae:.4f} eV")
print(f"  SchNet:  0.0670 eV  (published)")
print(f"  DimeNet: 0.0469 eV  (published)")

Test MAE: 0.1805 eV
Best val MAE: 0.1767 eV
Trained for: 91 epochs

Baseline comparison:
  Ours:    0.1805 eV
  SchNet:  0.0670 eV  (published)
  DimeNet: 0.0469 eV  (published)


In [15]:
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_100.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1  # 101
train_losses = checkpoint['train_losses']
val_losses = checkpoint['val_losses']
best_val_mae = min(val_losses)

print(f"Resuming from epoch {start_epoch}")
print(f"Best val MAE so far: {best_val_mae:.4f}")

Resuming from epoch 101
Best val MAE so far: 0.1767


In [16]:
for epoch in range(start_epoch, 201):  # 101 → 200
    train_mae = train_epoch(model, train_loader, optimizer, device)
    val_mae = evaluate(model, val_loader, device)
    scheduler.step(val_mae)

    train_losses.append(train_mae)
    val_losses.append(val_mae)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_mae': val_mae,
            'config': CONFIG,
        }, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))

    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_mae': val_mae,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'config': CONFIG,
        }, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pt'))
        print(f"Epoch {epoch:03d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f} | Best: {best_val_mae:.4f}")

Epoch 110 | Train MAE: 0.1806 | Val MAE: 0.1807 | Best: 0.1701
Epoch 120 | Train MAE: 0.1764 | Val MAE: 0.1675 | Best: 0.1675
Epoch 130 | Train MAE: 0.1757 | Val MAE: 0.1746 | Best: 0.1675
Epoch 140 | Train MAE: 0.1731 | Val MAE: 0.1698 | Best: 0.1675
Epoch 150 | Train MAE: 0.1714 | Val MAE: 0.1745 | Best: 0.1675
Epoch 160 | Train MAE: 0.1716 | Val MAE: 0.1710 | Best: 0.1675
Epoch 170 | Train MAE: 0.1714 | Val MAE: 0.1731 | Best: 0.1675
Epoch 180 | Train MAE: 0.1711 | Val MAE: 0.1712 | Best: 0.1675
Epoch 190 | Train MAE: 0.1703 | Val MAE: 0.1723 | Best: 0.1675
Epoch 200 | Train MAE: 0.1705 | Val MAE: 0.1718 | Best: 0.1675


```
def resume_training(checkpoint_path, model, optimizer):
    """Call this if your runtime dies mid-training."""
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])

    print(f"Resumed from epoch {checkpoint['epoch']}")
    print(f"Val MAE at checkpoint: {checkpoint['val_mae']:.4f}")

    return start_epoch, train_losses, val_losses

# To resume, uncomment and run:
start_epoch, train_losses, val_losses = resume_training(
     os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_100.pt'),
     model, optimizer
 )
```

In [20]:
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_mae = evaluate(model, test_loader, device)
print(f"Test MAE: {test_mae:.4f} eV")
print(f"Best val MAE: {checkpoint['val_mae']:.4f} eV")
print(f"Trained for: {checkpoint['epoch']} epochs")

# Published baselines on QM9 HOMO-LUMO gap:
print(f"\nBaseline comparison:")
print(f"  Ours:    {test_mae:.4f} eV")
print(f"  SchNet:  0.0670 eV  (published)")
print(f"  DimeNet: 0.0469 eV  (published)")

Test MAE: 0.1712 eV
Best val MAE: 0.1675 eV
Trained for: 120 epochs

Baseline comparison:
  Ours:    0.1712 eV
  SchNet:  0.0670 eV  (published)
  DimeNet: 0.0469 eV  (published)
